In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
from skopt.utils import OptimizeResult
import sys
sys.path.append('../../src')
from mfdt.config_finder.ff_helpers import SerialOptimizeResult
from mfdt.config_finder.ff_figures import (
    plot_optimisation_process,
    _plot_mesh,
    _plot_trajectory,
)

# Experiment 1&3

In [ ]:
EXPERIMENTS = [
    'exp_a',
    'exp_b',
    'exp_c',
    'exp_d',
    'exp_i',
]
NETWORK_TYPE = 'bigreal'
NETWORK_NAME = 'Freebase'

In [ ]:
for exp in EXPERIMENTS:
    serial_optim_result = SerialOptimizeResult.load(
        f'../../data/finder/{exp}/{NETWORK_TYPE}/{NETWORK_NAME}/logs/optim.pkl'
    )

    optim_result = OptimizeResult()
    optim_result.x_iters = serial_optim_result.x_iters
    optim_result.func_vals = serial_optim_result.func_vals

    optim_result.fun = serial_optim_result.fun
    optim_result.x = serial_optim_result.x

    save_path = Path(f'../../data/finder/{exp}/{NETWORK_TYPE}/{NETWORK_NAME}/logs/trajectory.png')


    plot_optimisation_process(
        result=optim_result,
        out_dir=save_path,
    )

# EXPERIMENT 2

In [ ]:
EXPERIMENTS = [
    'exp_e',
    'exp_f',
    'exp_g',
]
NETWORK_TYPE = 'bigreal'
NETWORK_NAME = 'Freebase'

In [ ]:
def plot_concat_optimisation_process(
    results: dict[str, OptimizeResult],
    out_dir: Path,
    exp: str,
) -> None:
    """Plot the optimisation process for a skopt result."""
    n_rows = math.ceil((len(results) + 1) / 2)
    fig, ax = plt.subplots(nrows=n_rows, ncols=2, figsize=(9, 4*n_rows))
    fig.suptitle("Optimisation Results")

    ax[0, 0] = _plot_mesh(ax[0, 0], results[exp])
    title_name = ' '.join(exp.upper().split('_'))
    ax[0, 0].set_title(f"Loss landscape & trajectory for {title_name}")

    remaining_exps = [e for e in EXPERIMENTS if e != exp]

    for i in range(ax.shape[0]):
        for j in range(ax.shape[1]):
            if i == 0 and j == 0:
                continue

            if i == 0:
                ax[i, j] = _plot_trajectory(ax[i, j], results[exp])
                title_name = ' '.join(exp.upper().split('_'))
            else:
                e = remaining_exps[0]
                remaining_exps.remove(e)
                ax[i, j] = _plot_trajectory(ax[i, j], results[e])
                title_name = ' '.join(e.upper().split('_'))

            ax[i, j].set_title(f"Convergence plot for {title_name}")

    assert len(remaining_exps) == 0

    fig.tight_layout()
    fig.savefig(out_dir, dpi=300)
    fig.savefig(out_dir.parent / "trajectory.png", dpi=300)
    fig.savefig(out_dir.parent / "trajectory.pdf", dpi=300, bbox_inches="tight", transparent=True,)

def load_otpim_Result(exp: str) -> OptimizeResult:
    serial_optim_result = SerialOptimizeResult.load(
        f'../../data/finder/{exp}/{NETWORK_TYPE}/{NETWORK_NAME}/logs/optim.pkl'
    )

    optim_result = OptimizeResult()
    optim_result.x_iters = serial_optim_result.x_iters
    optim_result.func_vals = serial_optim_result.func_vals

    optim_result.fun = serial_optim_result.fun
    optim_result.x = serial_optim_result.x

    return optim_result


In [ ]:
results = {exp: load_otpim_Result(exp) for exp in EXPERIMENTS}

for exp in EXPERIMENTS:
    save_path = Path(f'../../data/finder/{exp}/{NETWORK_TYPE}/{NETWORK_NAME}/logs/trajectory.png')

    plot_concat_optimisation_process(
        results=results,
        out_dir=save_path,
        exp=exp
    )

    result = results[exp]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax = _plot_trajectory(ax, result)
    fig.tight_layout()
    fig.savefig(save_path.parent / 'only_trajectory.png', dpi=300)
    fig.savefig(save_path.parent / 'only_trajectory.pdf', dpi=300, bbox_inches="tight", transparent=True,)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax = _plot_mesh(ax, result)
    fig.tight_layout()
    fig.savefig(save_path.parent / 'only_mesh.png', dpi=300)
    fig.savefig(save_path.parent / 'only_mesh.pdf', dpi=300, bbox_inches="tight", transparent=True,)

In [ ]:
file_name = 'trajectory_e_f_g'
for exp in EXPERIMENTS:
    save_path = Path(f'../../data/finder/{exp}/{NETWORK_TYPE}/{NETWORK_NAME}/logs')

    fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(14, 4))
    fig.suptitle("Optimisation Results")

    e = EXPERIMENTS[0]
    ax[0] = _plot_trajectory(ax[0], results[e])
    title_name = ' '.join(e.upper().split('_'))
    ax[0].set_title(f"Convergence plot for {title_name}")

    e = EXPERIMENTS[1]
    ax[1] = _plot_trajectory(ax[1], results[e])
    title_name = ' '.join(e.upper().split('_'))
    ax[1].set_title(f"Convergence plot for {title_name}")

    e = EXPERIMENTS[2]
    ax[2] = _plot_trajectory(ax[2], results[e])
    title_name = ' '.join(e.upper().split('_'))
    ax[2].set_title(f"Convergence plot for {title_name}")

    fig.tight_layout()
    fig.savefig(save_path / f"{file_name}.png", dpi=300)
    fig.savefig(save_path / f"{file_name}.pdf", dpi=300, bbox_inches="tight", transparent=True,)